In [1]:
import ROOT, glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

In [2]:
dirDir= "/home/users/l/lebrun/"
dataDir = "/dpnc/beegfs/dampe/tryf/MC_Skimmed/MC_gamma/"
figPath_ = dirDir + "figures/"
resultPath = dirDir + "results/"
dataPath = dirDir + "data/"
filename1 = dataDir + "DmlNtup_gamma_part*_TQ_FullSel.root"
filename2 = dataDir + "DmlNtup_gamma_100GeV_1TeV_part*_TQ_FullSel.root"
filename3 = dataDir + "DmlNtup_gamma_1TeV_10TeV_part*_MPhotonSelProj_TQ_FullSel.root"

filenames_LET = [filename1, filename2]
filenames_HET = [filename1, filename2, filename3]

# Find all matching ROOT files : from 1 GeV to 100 GeV
files = list()
for filename in filenames_HET:
    files += sorted(glob.glob(filename))
exponent = 0

print(f"Number of files found : {len(files)}")
print(files)

file = dataPath + f"MC_exp{exponent:.1f}.root"

trees = "DmlNtup"
tree = "MC"

# Pass the list of files to RDataFrame
rdf = ROOT.RDataFrame(tree, file)

# Apply cut on FoV
#rdf = rdf.Filter("theta_True < 60")


In [3]:
df = rdf.AsNumpy(["E_true", "tt_theta", "theta_BestSTK", 'tt_bgoTotalEcorr_GeV'])
df = pd.DataFrame(df)
print("Number of entries in original : ", len(df))

In [4]:
# Print min max of E_true
print(f"Minimum E_true: {df['E_true'].min()} GeV")
print(f"Maximum E_true: {df['E_true'].max()} GeV")

df['theta_True']= df['tt_theta']
df['theta_True_deg'] = np.rad2deg(df['theta_True'])
print(f"Minimum theta_True: {df['theta_True_deg'].min()} degrees")
print(f"Maximum theta_True: {df['theta_True_deg'].max()} degrees")

df['cosTheta'] = np.cos(df['theta_True'])
df['cosThetaSTK'] = np.cos(np.deg2rad(df['theta_BestSTK']))

df['E_reco'] = df["tt_bgoTotalEcorr_GeV"]

In [5]:
Emin = 1
Emax_HET = 1e4
n_E_bins_HET = 45 # 40

cos_theta_min = 0
cos_theta_max = 1
n_theta_bins = 20

E_bins_edge_HET = np.logspace(np.log10(Emin), np.log10(Emax_HET), n_E_bins_HET + 1)
E_bins_center_HET = np.sqrt(E_bins_edge_HET[:-1] * E_bins_edge_HET[1:])

theta_bins_edge = np.linspace(cos_theta_min, cos_theta_max, n_theta_bins + 1)
theta_bins_center = (theta_bins_edge[:-1] + theta_bins_edge[1:])/2

In [6]:
from scipy.optimize import curve_fit
try :
    n_bins = 50
    E_bins_edge_HET_ = np.logspace(np.log10(Emin), np.log10(Emax_HET), n_bins + 1)

    plt.figure(figsize=(8, 6))
    # Plot histogram for E_true
    counts, bins, _ = plt.hist(df['E_true'], bins=E_bins_edge_HET_, histtype='step', color='blue', label='E_true spectrum', weights=df['weight'])
    # Plot histogram for E_reco
    counts_reco, bins_reco, _ = plt.hist(df['E_reco'], bins=E_bins_edge_HET_, histtype='step', color='orange', label='E_reco spectrum', weights=df['weight'])

    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('Energy [GeV]')
    plt.ylabel('Number of events')
    plt.title('Energy Spectrum of $E_{true}$ and $E_{reco}$')

    # Fit the bin centers with a power law (for E_true)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    mask = counts > 0
    x = bin_centers[mask]
    y = counts[mask]

    def power_law(x, a, b):
        return a * x**b

    popt, pcov = curve_fit(power_law, x, y)
    a_fit, b_fit = popt

    # Plot the fit for E_true
    x_fit = np.logspace(np.log10(x.min()), np.log10(x.max()), 200)
    plt.plot(x_fit, power_law(x_fit, *popt), 'r--', label=f'E_true Fit: $a x^b$\n$b$={b_fit:.2f}')

    # Fit the bin centers with a power law (for E_reco), ignoring the first three bins
    bin_centers_reco = 0.5 * (bins_reco[:-1] + bins_reco[1:])
    mask_reco = counts_reco > 0
    # Ignore the first three bins
    x_reco = bin_centers_reco[mask_reco][5:]
    y_reco = counts_reco[mask_reco][5:]

    popt_reco, pcov_reco = curve_fit(power_law, x_reco, y_reco)
    a_fit_reco, b_fit_reco = popt_reco

    # Plot the fit for E_reco
    plt.plot(x_fit, power_law(x_fit, *popt_reco), 'g--', label=f'E_reco Fit: $a x^b$\n$b$={b_fit_reco:.2f}')

    plt.legend()
    plt.tight_layout()
    plt.show()


    plt.figure(figsize=(8, 6))
    # Plot histogram for E_true
    counts, bins, _ = plt.hist(df['E_true'], bins=E_bins_edge_HET_, histtype='step', color='blue', label='E_true spectrum')
    # Plot histogram for E_reco
    counts_reco, bins_reco, _ = plt.hist(df['E_reco'], bins=E_bins_edge_HET_, histtype='step', color='orange', label='E_reco spectrum')

    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('Energy [GeV]')
    plt.ylabel('Number of events')
    plt.title('Energy Spectrum of $E_{true}$ and $E_{reco}$')
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as e :
    print(f"Error occurred: {e}")
    import traceback
    traceback.print_exc()

In [7]:
def normalize_by_energy_bin(counts):
    """Normalize each energy bin (column) by its total count"""
    print(f"Input counts shape: {counts.shape}")
    print(f"Input counts range: {counts.min()} to {counts.max()}")
    
    energy_totals = np.sum(counts, axis=1, keepdims=True)
    print(f"Energy totals shape: {energy_totals.shape}")
    print(f"Energy totals range: {energy_totals.min()} to {energy_totals.max()}")
    
    normalized = np.divide(counts, energy_totals, out=np.zeros_like(counts), where=energy_totals!=0)
    print(f"Normalized shape: {normalized.shape}")
    print(f"Normalized range: {normalized.min()} to {normalized.max()}")
    
    return normalized


In [10]:
import os
# Debug: Check your variables first
print("=== DEBUGGING INFO ===")
try:
    print(f"df shape: {df.shape}")
    print(f"df columns: {list(df.columns)}")
    print(f"E_bins_edge_HET shape: {E_bins_edge_HET.shape}")
    print(f"theta_bins_edge shape: {theta_bins_edge.shape}")
    
    # Check for required columns
    required_cols = ['E_true', 'cosTheta', 'cosThetaSTK']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"ERROR: Missing columns: {missing_cols}")
        print(f"Available columns: {list(df.columns)}")
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Check for NaN values
    for col in required_cols:
        nan_count = df[col].isna().sum()
        if nan_count > 0:
            print(f"WARNING: {col} has {nan_count} NaN values")
    
    # Check data ranges
    print(f"E_true range: {df['E_true'].min()} to {df['E_true'].max()}")
    print(f"cosTheta range: {df['cosTheta'].min()} to {df['cosTheta'].max()}")
    print(f"cosThetaSTK range: {df['cosThetaSTK'].min()} to {df['cosThetaSTK'].max()}")
    
except Exception as e:
    print(f"Error in debugging: {e}")
    traceback.print_exc()


In [ ]:

print("=== STARTING PLOTS ===")

vmin = 0
vmax = 0.2



try:
    # Create figure with explicit size
    fig, axs = plt.subplots(1, 2, figsize=(16, 8), sharey=True)
    
    # Left plot - Total, True theta
    print("Creating plot 1...")
    
    # Check for valid data before histogramming
    valid_mask1 = ~(np.isnan(df['E_true']) | np.isnan(df['cosTheta']))
    print(f"Valid data points for plot 1: {valid_mask1.sum()} out of {len(df)}")
    
    if valid_mask1.sum() == 0:
        raise ValueError("No valid data points for plot 1")
    
    counts1, x_edges1, y_edges1 = np.histogram2d(
        df.loc[valid_mask1, 'E_true'], 
        df.loc[valid_mask1, 'cosTheta'], 
        bins=[E_bins_edge_HET, theta_bins_edge]
    )
    
    print(f"counts1 shape: {counts1.shape}")
    print(f"counts1 total: {counts1.sum()}")
    
    if counts1.sum() == 0:
        print("WARNING: All histogram bins are empty for plot 1")
    
    normalized1 = normalize_by_energy_bin(counts1)
    normalized1_masked = np.ma.masked_where(normalized1 <= 0, normalized1)
    
    im1 = axs[0].pcolormesh(E_bins_edge_HET, theta_bins_edge, normalized1_masked.T, 
                            cmap='viridis', vmin=vmin, vmax=vmax)
    
    # Add colorbar with error handling
    try:
        fig.colorbar(im1, ax=axs[0], label="Fraction of events per energy bin")
    except Exception as cb_error:
        print(f"Colorbar error for plot 1: {cb_error}")
    
    axs[0].set_xlabel('True Energy [GeV]')
    axs[0].set_xscale('log')
    axs[0].set_ylabel('True cos(θ)')
    axs[0].set_title('Total, True θ - Angular Distribution')
    
    # Right plot - Accepted, True theta
    print("Creating plot 2...")
    
    valid_mask2 = ~(np.isnan(df['E_true']) | np.isnan(df['cosThetaSTK']))
    print(f"Valid data points for plot 2: {valid_mask2.sum()} out of {len(df)}")
    
    if valid_mask2.sum() == 0:
        raise ValueError("No valid data points for plot 2")
    
    counts2, x_edges2, y_edges2 = np.histogram2d(
        df.loc[valid_mask2, 'E_true'], 
        df.loc[valid_mask2, 'cosThetaSTK'], 
        bins=[E_bins_edge_HET, theta_bins_edge]
    )
    
    print(f"counts2 shape: {counts2.shape}")
    print(f"counts2 total: {counts2.sum()}")
    
    if counts2.sum() == 0:
        print("WARNING: All histogram bins are empty for plot 2")
    
    normalized2 = normalize_by_energy_bin(counts2)
    normalized2_masked = np.ma.masked_where(normalized2 <= 0, normalized2)
    
    im2 = axs[1].pcolormesh(E_bins_edge_HET, theta_bins_edge, normalized2_masked.T, 
                            cmap='viridis', vmin=vmin, vmax=vmax)
    
    try:
        fig.colorbar(im2, ax=axs[1], label="Fraction of events per energy bin")
    except Exception as cb_error:
        print(f"Colorbar error for plot 2: {cb_error}")
    
    axs[1].set_xlabel('True Energy [GeV]')
    axs[1].set_xscale('log')
    axs[1].set_ylabel('True cos(θ)')
    axs[1].set_title('Accepted, True θ - Angular Distribution')

    fig.suptitle("Angular distribution - Unweighted", size='large')
    
    plt.tight_layout()
    plt.show()
    print("Plotting completed successfully!")
    
    # Print some statistics
    if counts1.sum() > 0:
        print(f"Plot 1 - Energy bin 10 normalization check:")
        if len(normalized1) > 10:
            print(f"Sum of normalized counts in energy bin 10: {np.sum(normalized1[10, :]):.3f}")
    
except Exception as e:
    print(f"Error occurred: {e}")
    traceback.print_exc()
    
    # Additional debugging info
    print("\n=== ADDITIONAL DEBUG INFO ===")
    try:
        print(f"Current working directory: {os.getcwd()}")
        print(f"Matplotlib backend: {plt.get_backend()}")
        print(f"Figure list: {plt.get_fignums()}")
    except:
        pass